# 🪴 Training Notebook for Plant Leaf Detection

This notebook demonstrates how to prepare data, build a model, and train a plant leaf detection/classification system. You can adapt the code to your dataset and export the trained classifier to `saved_models/leaf_model.pkl` for use in the Streamlit application.

## 1. Import Required Libraries

We'll import the common libraries used for image processing and model training.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# optional deep learning frameworks
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# utilities for augmentation
import cv2

print("Libraries imported successfully")

## 2. Load and Explore Dataset

Load your leaf images and any corresponding labels/annotations. Display a few examples and check class balance.

In [ ]:
# adjust this path to where your dataset lives
DATA_DIR = os.path.abspath(os.path.join('..', 'data', 'leaf_dataset'))

# example structure: DATA_DIR/species_name/image1.jpg
classes = [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))]
print('Found classes:', classes)

# show a few sample images
for cls in classes[:3]:
    imgs = os.listdir(os.path.join(DATA_DIR, cls))[:2]
    for img in imgs:
        display(Image.open(os.path.join(DATA_DIR, cls, img)).resize((200, 200)))

# optionally create a DataFrame of paths and labels
records = []
for cls in classes:
    for fname in os.listdir(os.path.join(DATA_DIR, cls)):
        records.append((os.path.join(DATA_DIR, cls, fname), cls))

df = pd.DataFrame(records, columns=['path', 'label'])
print(df['label'].value_counts())

## 3. Preprocess Images

Resize to a consistent shape, normalize pixel values and perform augmentation when necessary.

In [ ]:
IMAGE_SIZE = (128, 128)

# simple preprocessing function
def preprocess(path):
    img = Image.open(path).convert('RGB')
    img = img.resize(IMAGE_SIZE)
    arr = np.asarray(img) / 255.0
    return arr

# prepare numpy arrays for training (this is just one simple way)
data = np.stack(df['path'].apply(preprocess).to_list())
labels = pd.get_dummies(df['label']).values

print('data shape', data.shape, 'labels shape', labels.shape) 

## 4. Build Detection Model

Here we'll define a simple CNN classifier. Replace with a more advanced object‑detection architecture if you need bounding boxes instead of just a class label.

In [ ]:
model = models.Sequential([
    layers.Input(shape=(*IMAGE_SIZE, 3)),
    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(len(classes), activation='softmax')
])
model.summary()

## 5. Compile and Train the Model

Compile with an appropriate loss and optimizer, and fit using a validation split.

In [ ]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

history = model.fit(
    data, labels,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    callbacks=[callbacks.EarlyStopping(patience=3, restore_best_weights=True)]
)  

## 6. Evaluate Model Performance

Plot training history and evaluate on held‑out data.

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(history.history['accuracy'], label='train acc')
plt.plot(history.history['val_accuracy'], label='val acc')
plt.legend();

# if you kept a separate test set, you could evaluate here
# test_loss, test_acc = model.evaluate(test_data, test_labels)

## 7. Run Inference on New Images

Load new leaf images and run the model to see predictions.

In [ ]:
sample_path = df['path'].iloc[0]
img = preprocess(sample_path)
pred = model.predict(np.expand_dims(img, axis=0))
print('predicted class:', classes[np.argmax(pred)])
display(Image.open(sample_path))

## 8. Save and Load the Model

After training, save the model so the Streamlit application can load it using `joblib` or the Keras API.

In [ ]:
export_path = os.path.abspath(os.path.join('..', 'saved_models', 'leaf_model.pkl'))

# for scikit-learn style serialisation you could do:
# import joblib
# joblib.dump(model, export_path)

# for keras model (recommended if using tf):
model.save(os.path.join('..','saved_models','leaf_model'))

print('Model saved')